In [ ]:
import math
import random
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from part1.ols_implementation import ols_fit, coef_inference, vif
from part1.gauss_markov_demo import run_gauss_markov_simulation, calculate_stats
from part1.ols_implementation import make_collinear_data 

# ===========================================================================
# CHUẨN BỊ DỮ LIỆU GỐC & FIT MÔ HÌNH OLS (Phục vụ cho F4)
# ===========================================================================
random.seed(42)
X = []
y = []
for _ in range(50):
    x1 = random.gauss(10, 2)
    x2 = random.gauss(5, 1) + 0.5 * x1  # Có tương quan nhẹ nhưng chưa bị đa cộng tuyến
    e = random.gauss(0, 0.5)
    y_val = 2.0 + 1.5 * x1 - 0.8 * x2 + e
    X.append([x1, x2])
    y.append(y_val)

# Chạy mô hình OLS gốc từ hàm tự viết để lấy kết quả nền tảng
ols_result = ols_fit(X, y)
beta_hat = ols_result['beta_hat']
sigma2 = ols_result['sigma2']


# ===========================================================================
# CHUẨN BỊ DỮ LIỆU ĐA CỘNG TUYẾN (Phục vụ cho F5)
# ===========================================================================
# Gọi hàm tạo dữ liệu collinear theo đúng "KPI" trong ảnh yêu cầu
X_collinear = make_collinear_data()


In [ ]:
# ---------------------------------------------------------------------------
# Visualize F10: Mô phỏng Monte Carlo & Chứng minh định lý Gauss-Markov (BLUE)
# ---------------------------------------------------------------------------

def visualize_results(beta_ols, beta_alt, true_beta):
    cols = ['intercept', 'x1', 'x2']
    df_ols = pd.DataFrame(beta_ols, columns=cols)
    df_alt = pd.DataFrame(beta_alt, columns=cols)
    
    # 1. Vẽ Histogram cho từng hệ số j theo đúng yêu cầu
    for i, col in enumerate(cols):
        fig, ax = plt.subplots(figsize=(8, 5))
        
        # Vẽ phân phối OLS (Ước lượng BLUE - gầy và cao)
        ax.hist(df_ols[col], bins=40, alpha=0.6, label='OLS Estimator (BLUE)', color='skyblue', edgecolor='black')
        
        # Vẽ phân phối của Estimator khác để đối chứng (Alt Estimator - thấp và bè)
        ax.hist(df_alt[col], bins=40, alpha=0.4, label='Alternative Estimator', color='salmon', edgecolor='gray')
        
        # Đường thẳng đứng tại giá trị TRUE_BETA[j]
        ax.axvline(true_beta[i], color='red', linestyle='--', linewidth=2.5, label=f'True β = {true_beta[i]}')
        
        # Cấu hình hiển thị chuẩn chỉnh
        ax.set_title(f"Monte Carlo Distribution & Variance Comparison for {col}", fontsize=13)
        ax.set_xlabel("Value of Beta Hat", fontsize=11)
        ax.set_ylabel("Frequency", fontsize=11)
        ax.legend(loc='upper right')
        ax.grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        # Lưu file chuẩn không chứa khoảng trắng gây lỗi hệ thống
        plt.savefig(f"output/monte_carlo_{col}.png", dpi=150, bbox_inches='tight')
        plt.show()

    # 2. Bảng so sánh Mean(β) và Var(β) giữa OLS và Estimator khác để chứng minh BLUE
    comparison = pd.DataFrame({
        'True Beta': true_beta,
        'Mean (OLS)': df_ols.mean().values,       # Phải xấp xỉ True Beta (Không chệch)
        'Var (OLS)': df_ols.var().values,         # Phải nhỏ nhất (Tính chất Best/Efficiency)
        'Var (Alternative)': df_alt.var().values  # Phải lớn hơn Var OLS
    }, index=cols)
    
    return comparison

# ===========================================================================
# TIẾN HÀNH CHẠY THỰC THI CELL F10
# ===========================================================================
# 1. Gọi hàm sinh dữ liệu mô phỏng
beta_ols_sim, beta_alt_sim, true_beta_config = run_gauss_markov_simulation()

# 2. Thực thi vẽ đồ thị và nhận về bảng thống kê đối chứng
comparison_table = visualize_results(beta_ols_sim, beta_alt_sim, true_beta_config)

print("\n--- BẢNG SO SÁNH KIỂM CHỨNG ĐỊNH LÝ GAUSS-MARKOV ---")
display(comparison_table)

In [ ]:
# ---------------------------------------------------------------------------
# Visualize F4: Coefficient Inference Table with Highlights
# ---------------------------------------------------------------------------

df_f4_results = coef_inference(X, y, beta_hat, sigma2)

def highlight_p_value(val):
    return 'background-color: #fffa9e; color: black; font-weight: bold' if val < 0.05 else ''

print("--- [KẾT QUẢ TỰ VIẾT] BẢNG SUY DIỄN HỆ SỐ OLS (F4) ---")
display(df_f4_results.style.map(highlight_p_value, subset=['p_value']))

print("\n--- [ĐỐI CHỨNG] KẾT QUẢ TỪ THƯ VIỆN STATSMODELS ---")

# 1. Thêm cột hằng số (intercept) vào ma trận X theo đúng quy chuẩn của statsmodels
X_with_intercept = sm.add_constant(X)

# 2. Khởi tạo mô hình OLS từ thư viện chuẩn và tiến hành fit dữ liệu
sm_model = sm.OLS(y, X_with_intercept)
sm_results = sm_model.fit()

# 3. In bảng summary chi tiết để đối chứng trực quan
print(sm_results.summary())

In [ ]:
# ---------------------------------------------------------------------------
# Visualize F5: Variance Inflation Factor (VIF) & Multicollinearity Check
# ---------------------------------------------------------------------------

# Giả sử hàm make_collinear_data() đã được định nghĩa hoặc import ở các cell trên
# Nó sẽ trả về ma trận dữ liệu độc lập X_collinear chứa đa cộng tuyến
X_collinear = make_collinear_data() 

# Tính toán hệ số VIF bằng hàm tự viết 
vif_custom = vif(X_collinear)

# Chuyển đổi thành DataFrame để hiển thị dạng bảng sạch sẽ
df_vif_custom = pd.DataFrame(list(vif_custom.items()), columns=['Biến độc lập', 'VIF (Tự viết)'])

print("--- [KẾT QUẢ TỰ VIẾT] BẢNG KIỂM TRA ĐA CỘNG TUYẾN VIF (F5) ---")
display(df_vif_custom)

# Vòng lặp kiểm tra điều kiện biên và in cảnh báo trực quan nếu VIF > 10
warning_triggered = False
for _, row in df_vif_custom.iterrows():
    if row['VIF (Tự viết)'] > 10:
        print(f"CẢNH BÁO NGUY HIỂM: Biến '{row['Biến độc lập']}' có VIF = {row['VIF (Tự viết)']:.2f} (> 10). Hệ thống bị đa cộng tuyến nặng!")
        warning_triggered = True

if not warning_triggered:
    print("Hệ thống an toàn: Tất cả các biến đều có VIF <= 10.")


print("\n--- [ĐỐI CHỨNG] KẾT QUẢ TỪ THƯ VIỆN STATSMODELS ---")

# statsmodels yêu cầu dữ liệu đầu vào phải ở dạng DataFrame hoặc mảng NumPy
# Chúng ta đưa dữ liệu về DataFrame để tính toán đồng bộ nhãn biến
df_sm_input = pd.DataFrame(X_collinear)

vif_statsmodels = {}
for i, col in enumerate(df_sm_input.columns):
    # Gọi chính xác hàm được yêu cầu trong ảnh demo
    vif_val = variance_inflation_factor(df_sm_input.values, i)
    # Gán nhãn tương ứng (ví dụ: x1, x2 hoặc theo tên key của vif_custom)
    var_name = list(vif_custom.keys())[i] if i < len(vif_custom) else f"x{i+1}"
    vif_statsmodels[var_name] = vif_val

# Hiển thị bảng đối chứng của thư viện
df_vif_sm = pd.DataFrame(list(vif_statsmodels.items()), columns=['Biến độc lập', 'VIF (Statsmodels)'])
display(df_vif_sm)